In [1]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError("GPU is not available. Go to Runtime > Change runtime type > GPU.")

CUDA available: True
GPU: Tesla T4


In [ ]:
# Mount Google Drive so checkpoints and the final model are saved there
# (safe to re-run – it will just re-mount if already mounted)
from google.colab import drive
import os

drive.mount('/content/drive', force_remount=False)

# ── Paths on Drive ──────────────────────────────────────────────
DRIVE_BASE      = '/content/drive/MyDrive/qwen-moroccan-law'
DRIVE_CKPT_DIR  = f'{DRIVE_BASE}/checkpoints'   # live checkpoints every N steps
DRIVE_ADAPTER   = f'{DRIVE_BASE}/lora-adapter'  # final LoRA weights
DRIVE_MERGED    = f'{DRIVE_BASE}/merged-model'  # optional: merged 16-bit model

os.makedirs(DRIVE_CKPT_DIR, exist_ok=True)
os.makedirs(DRIVE_ADAPTER, exist_ok=True)

print('Drive mounted. Saving to:', DRIVE_BASE)


In [2]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes

In [3]:
from google.colab import files
from pathlib import Path
import shutil

Path("data").mkdir(exist_ok=True)

uploaded = files.upload()

uploaded_filename = list(uploaded.keys())[0]
raw_path = "data/raw_qa.jsonl"

shutil.move(uploaded_filename, raw_path)

print(f"Uploaded file saved to: {raw_path}")

Saving raw_qa.jsonl to raw_qa.jsonl
Uploaded file saved to: data/raw_qa.jsonl


In [4]:
from pathlib import Path
import json
import random
from transformers import AutoTokenizer
from collections import Counter


MODEL_NAME = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit"

SYSTEM_PROMPT = (
    "You are a legal assistant specialized in Moroccan law. "
    "Answer in the same language as the user's question unless the user asks for another language. "
    "Answer clearly and accurately. "
    "When possible, cite the relevant law, article number, and legal source. "
    "If the answer is uncertain or the legal source is missing, say so. "
    "Your answer is for legal information only and does not replace professional legal advice."
)


def load_raw_jsonl(path: Path):
    rows = []

    with path.open("r", encoding="utf-8") as f:
        for line_number, line in enumerate(f, start=1):
            line = line.strip()

            if not line:
                continue

            try:
                item = json.loads(line)
            except json.JSONDecodeError as e:
                raise ValueError(f"Invalid JSON on line {line_number}: {e}")

            if "input" not in item or "output" not in item:
                raise ValueError(
                    f"Line {line_number} must contain 'input' and 'output'."
                )

            item["input"] = str(item["input"]).strip()
            item["output"] = str(item["output"]).strip()
            item["language"] = item.get("language", "unknown")

            if not item["input"] or not item["output"]:
                raise ValueError(f"Line {line_number} has empty input or output.")

            rows.append(item)

    return rows


def convert_to_qwen_chat_text(item, tokenizer):
    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content": item["input"],
        },
        {
            "role": "assistant",
            "content": item["output"],
        },
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )

    return {
        "text": text,
        "language": item.get("language", "unknown"),
    }


def save_jsonl(rows, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)

    with path.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")


raw_path = Path("data/raw_qa.jsonl")
train_path = Path("data/train.jsonl")
validation_path = Path("data/validation.jsonl")
formatted_path = Path("data/formatted_qa.jsonl")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
)

raw_rows = load_raw_jsonl(raw_path)

print("Total raw examples:", len(raw_rows))
print("Raw language distribution:", Counter(row["language"] for row in raw_rows))

random.seed(42)
random.shuffle(raw_rows)

formatted_rows = [
    convert_to_qwen_chat_text(row, tokenizer)
    for row in raw_rows
]

# Save full formatted dataset too, useful for debugging
save_jsonl(formatted_rows, formatted_path)

# Correct split: 90% train, 10% validation
split_index = int(len(formatted_rows) * 0.9)

train_rows = formatted_rows[:split_index]
validation_rows = formatted_rows[split_index:]

save_jsonl(train_rows, train_path)
save_jsonl(validation_rows, validation_path)

print("Formatting completed.")
print("Train examples:", len(train_rows))
print("Validation examples:", len(validation_rows))
print("Train language distribution:", Counter(row["language"] for row in train_rows))
print("Validation language distribution:", Counter(row["language"] for row in validation_rows))

print("\nFirst formatted train example:")
print(train_rows[0]["text"][:2000])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

Total raw examples: 5312
Raw language distribution: Counter({'unknown': 5312})
Formatting completed.
Train examples: 4780
Validation examples: 532
Train language distribution: Counter({'unknown': 4780})
Validation language distribution: Counter({'unknown': 532})

First formatted train example:
<|im_start|>system
You are a legal assistant specialized in Moroccan law. Answer in the same language as the user's question unless the user asks for another language. Answer clearly and accurately. When possible, cite the relevant law, article number, and legal source. If the answer is uncertain or the legal source is missing, say so. Your answer is for legal information only and does not replace professional legal advice.<|im_end|>
<|im_start|>user
ما هي المبادئ الأساسية التي تقوم عليها إعادة هيكلة القانون الشخصي في المغرب؟<|im_end|>
<|im_start|>assistant
تقوم على مبادئ العدالة والتعامل الإيجابي مع حقوق المرأة، وحماية حقوق الطفل، وحفظ كرامة الإنسان، وبناء مجتمع تطبيقاً للمبادئ الإسلامية من المس

In [5]:
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files={
        "train": "data/train.jsonl",
        "validation": "data/validation.jsonl",
    },
)

print(dataset)
print("Train size:", len(dataset["train"]))
print("Validation size:", len(dataset["validation"]))

print("\nFirst train example:")
print(dataset["train"][0]["text"][:1500])

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'language'],
        num_rows: 4780
    })
    validation: Dataset({
        features: ['text', 'language'],
        num_rows: 532
    })
})
Train size: 4780
Validation size: 532

First train example:
<|im_start|>system
You are a legal assistant specialized in Moroccan law. Answer in the same language as the user's question unless the user asks for another language. Answer clearly and accurately. When possible, cite the relevant law, article number, and legal source. If the answer is uncertain or the legal source is missing, say so. Your answer is for legal information only and does not replace professional legal advice.<|im_end|>
<|im_start|>user
ما هي المبادئ الأساسية التي تقوم عليها إعادة هيكلة القانون الشخصي في المغرب؟<|im_end|>
<|im_start|>assistant
تقوم على مبادئ العدالة والتعامل الإيجابي مع حقوق المرأة، وحماية حقوق الطفل، وحفظ كرامة الإنسان، وبناء مجتمع تطبيقاً للمبادئ الإسلامية من المساواة والعدل والتسامح.<|im_end|>


In [6]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-3B-Instruct-bnb-4bit",
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
)


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/usr/local/lib/python3.12/dist-packages/unsloth/__init__.py:127: UserWarning: WARNING: Unsloth should be imported before [transformers] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *


🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.5.2: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.05G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

unsloth/Qwen2.5-3B-Instruct-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


In [7]:
model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_alpha=64,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
    use_rslora=False,
    loftq_config=None,
)

print("LoRA adapters added.")

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.5.2 patched 36 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


LoRA adapters added.


In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported
import os

# Checkpoint dir: save directly to Google Drive every save_steps
# so you never lose more than save_steps worth of progress on disconnect.
DRIVE_CKPT_DIR = '/content/drive/MyDrive/qwen-moroccan-law/checkpoints'
os.makedirs(DRIVE_CKPT_DIR, exist_ok=True)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=max_seq_length,

        per_device_train_batch_size=1,
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=16,

        num_train_epochs=3,
        learning_rate=2e-4,
        warmup_steps=5,
        weight_decay=0.01,

        logging_steps=5,
        eval_steps=25,
        save_steps=25,         # checkpoint written to Drive every 25 steps

        eval_strategy="steps",
        save_strategy="steps",
        save_total_limit=3,    # keep last 3 checkpoints on Drive

        optim="adamw_8bit",
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),

        report_to="none",
        packing=False,
        output_dir=DRIVE_CKPT_DIR,  # <── checkpoints go straight to Drive
    ),
)

train_result = trainer.train()
print(train_result)


In [ ]:
import shutil, os

# ── Paths ────────────────────────────────────────────────────────
LOCAL_ADAPTER  = 'qwen-moroccan-law-lora'
DRIVE_ADAPTER  = '/content/drive/MyDrive/qwen-moroccan-law/lora-adapter'
DRIVE_MERGED   = '/content/drive/MyDrive/qwen-moroccan-law/merged-model'

# 1. Save LoRA adapter locally (fast, used for inference below)
model.save_pretrained(LOCAL_ADAPTER)
tokenizer.save_pretrained(LOCAL_ADAPTER)
print(f'Adapter saved locally to: {LOCAL_ADAPTER}')

# 2. Copy LoRA adapter to Google Drive (survives disconnect)
os.makedirs(DRIVE_ADAPTER, exist_ok=True)
shutil.copytree(LOCAL_ADAPTER, DRIVE_ADAPTER, dirs_exist_ok=True)
print(f'Adapter copied to Drive: {DRIVE_ADAPTER}')

# 3. (Optional) Save full merged 16-bit model to Drive
#    Uncomment if you want a standalone model that doesn't need the base weights.
#    NOTE: this uses ~6 GB of Drive space for a 3B model.
# model.save_pretrained_merged(
#     DRIVE_MERGED,
#     tokenizer,
#     save_method='merged_16bit',
# )
# print(f'Merged model saved to Drive: {DRIVE_MERGED}')

print('\nAll done! Your model is safe on Google Drive.')


In [10]:
from unsloth import FastLanguageModel

max_seq_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="qwen-moroccan-law-lora",
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
)

FastLanguageModel.for_inference(model)

print("Fine-tuned adapter loaded for inference.")

==((====))==  Unsloth 2026.5.2: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

unsloth/Qwen2.5-3B-Instruct-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
Fine-tuned adapter loaded for inference.


In [11]:
SYSTEM_PROMPT = (
    "You are a legal assistant specialized in Moroccan law. "
    "Answer in the same language as the user's question unless the user asks for another language. "
    "Answer clearly and accurately. "
    "When possible, cite the relevant law, article number, and legal source. "
    "If the answer is uncertain or the legal source is missing, say so. "
    "Your answer is for legal information only and does not replace professional legal advice."
)


def ask_model(question, max_new_tokens=300):
    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content": question,
        },
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(
        [prompt],
        return_tensors="pt",
    ).to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        repetition_penalty=1.1,
        pad_token_id=tokenizer.eos_token_id,
    )

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True,
    )

    return response.strip()

In [12]:
test_questions = [
    "What is considered an abandoned child according to Article 1 of Law 15-01?",
    "Qu'est-ce qu'un enfant abandonné selon l'article 1 de la loi 15-01 ?",
    "ما هو الطفل المهمل حسب المادة 1 من القانون 15-01؟",
]

for question in test_questions:
    print("=" * 100)
    print("QUESTION:")
    print(question)
    print("\nANSWER:")
    print(ask_model(question))

Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


QUESTION:
What is considered an abandoned child according to Article 1 of Law 15-01?

ANSWER:


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


According to Article 1, an abandoned child is any child who has reached the age of majority but is still subject to kafala (a system of care and protection for children), or a child whose parents have died or are unable to provide for their needs due to indigence or other exceptional circumstances.
QUESTION:
Qu'est-ce qu'un enfant abandonné selon l'article 1 de la loi 15-01 ?

ANSWER:


Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Un enfant abandonné est un enfant né de parents inconnus ou de parents connus qui ont abdiqué volontairement leur responsabilité envers cet enfant, ou un enfant issu d'une relation illégale dont les parents sont démunis ou incapables de subvenir à ses besoins.
QUESTION:
ما هو الطفل المهمل حسب المادة 1 من القانون 15-01؟

ANSWER:
الطفل المهمل هو الطفل الذي لم يبلغ السابعة من عمره، ويعيش مع أحد والديه أو شخص آخر دون أن يكون لهما مسؤولية عن رعايته، ولا يوجد لهما ما يسمى بالورثة، ويكون قد تعرض للعنف أو الإهمال من قبل أحد الأشخاص المذكورين أو غيرهم، ويكون قد تقرر أن يكون لهما مسؤولية عن رعايته.
